# TweetToxic 2026 PUJ · Notebook final (equipo NLP_G1)
Curso de Procesamiento del Lenguaje Natural, Maestría en Inteligencia Artificial, Pontificia Universidad Javeriana.

Este notebook reúne de principio a fin la solución entregada:
1. Análisis de datos.
2. Preprocesamiento.
3. Variables lingüísticas.
4. Diseño de validación.
5. Comparación de alternativas.
6. Validación en un conjunto de control.
7. Análisis de errores.
8. Generación de las dos submissions finales:
   - **E11c**: stacking con *seed bagging*, sin variables de contexto nuevas.
   - **E13a**: lo mismo, más las variables de contexto H12-H17 (versión genérica) y H19-H23.

El historial completo de los 14 experimentos (E0-E13), con sus notebooks y scripts, está en `INFORME_EXPERIMENTOS.md` y `RESUMEN.md`.

**Cómo ejecutarlo.**
- Desde la carpeta `experimentos/`, con el entorno de `../requirements.txt` y los datos de Kaggle en `../tweet-toxic-2026-puj/` (o en `/kaggle/input/tweet-toxic-2026-puj`).
- Las probabilidades de cada modelo base se guardan en `oof/`. Con esa caché el notebook corre en pocos minutos. Sin ella, reentrena todo: unas 3 horas en CPU, porque son 7 modelos × 5 semillas × 5 particiones por cada conjunto de variables.
- **El test nunca entra en un `fit`**: vectorizadores, escaladores, SVD y modelos se ajustan solo con la partición de entrenamiento de cada fold, o con el train completo para el test.

### Preparación en Kaggle
Este notebook usa el dataset privado `tweettoxic-nlp-g1-codigo`, que tiene el código (`toxic_base.py`, `ext_*.py`), el conjunto de control, la caché `oof/` y las dos submissions enviadas. También usa los datos de la competencia. `/kaggle/input` es de solo lectura, así que se copia todo a `/kaggle/working/experimentos` y se trabaja desde ahí.

In [ ]:
import os, glob, shutil
base = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/toxic_base.py', recursive=True))
shutil.copytree(base, '/kaggle/working/experimentos', dirs_exist_ok=True)
datos = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/toxic_train.csv', recursive=True))
if not os.path.exists('/kaggle/working/tweet-toxic-2026-puj'):
    os.symlink(datos, '/kaggle/working/tweet-toxic-2026-puj')
# submission original (0,79160): guardar_submission la usa como referencia para contar filas cambiadas
shutil.copy('/kaggle/working/experimentos/toxic_submission_original.csv', '/kaggle/working/toxic_submission.csv')
os.chdir('/kaggle/working/experimentos')
print('Código desde', base, '| datos desde', datos, '| trabajando en', os.getcwd())

In [ ]:
import os, sys, platform, re
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, pandas as pd, sklearn, scipy, lightgbm
import toxic_base as tb, ext_variables as ev, ext_variables2 as ev2, ext_formulaciones as ef
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 140)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__, '| scikit-learn',
      sklearn.__version__, '| scipy', scipy.__version__, '| lightgbm', lightgbm.__version__)
SEEDS_CV = tb.SEEDS                       # semillas de la CV repetida (evaluación)
S5 = (42, 43, 44, 45, 46)                 # semillas del seed bagging (modelos base)
B5 = tb.BASES_ORIGINALES                  # CNB, LR, SVM, NB-LR, Powerset
B7 = B5 + ['LGBM', 'LGBM-PS']
print('Semillas CV:', SEEDS_CV, '| semillas bagging:', S5, '| particiones:', tb.N_FOLDS)
print('Configuración base:', tb.CFG_BASE)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

## 1. Análisis de datos
Tres etiquetas binarias:
- **HS**: discurso de odio contra mujeres o inmigrantes.
- **TR**: el odio va dirigido a una persona concreta.
- **AG**: el tweet es agresivo.

El corpus sale de HatEval 2019 (SemEval-2019, tarea 5), en español.

In [ ]:
print('Prevalencia (proporción de 1):'); display(d.train[tb.LABELS].mean().round(3).to_frame('train').T)
print('Combinaciones HS-TR-AG en el train:'); display(pd.Series(d.COMBO).value_counts().to_frame('n').T)
print('¿TR=1 implica HS=1 siempre?', bool(((d.Y[:, 1] == 1) <= (d.Y[:, 0] == 1)).all()))
largo = d.train['text'].str.split().str.len()
print('Palabras por tweet: mediana', int(largo.median()), '| p95', int(largo.quantile(.95)))

### Artefactos de los datos
- **Se perdieron las tildes y la ñ.** Los caracteres no ASCII se borraron en el train (*cállate* → `cllate`).
- **Celdas con el error de Excel `#¿NOMBRE?`.** Hay 1 en el train y 6 en el test.
- **Texto entre comillas.** En el train, todas las filas entrecomilladas tienen AG = 1.

In [ ]:
no_ascii = lambda s: s.str.contains(r'[^\x00-\x7f]').mean()
print('Filas con caracteres no ASCII | train:', round(no_ascii(d.train['text']), 3), '| test:', round(no_ascii(d.test['text']), 3))
print('Ejemplos de palabras sin vocal acentuada:', sorted({w for t in d.TXT_TR for w in re.findall(r'\b(cllate|imbcil|est|ests|aqu|qu)\b', t)}))
err_tr = d.train['text'].str.strip().str.match(tb.RE_ERROR_EXCEL)
print('#¿NOMBRE? | train:', int(err_tr.sum()), '| test:', int(d.err_excel_te.sum()))
com = d.train['text'].str.contains('"', regex=False)
print('Filas entrecomilladas en train:', int(com.sum()), '| con AG=1:', int(d.train.loc[com, 'AG'].sum()))

## 2. Preprocesamiento
La función `tb.normalizar`:
- Pasa todo a minúsculas.
- Cambia enlaces y menciones por `xurl` y `xuser`.
- Borra los caracteres no ASCII también en el test, para que quede igual que el train.
- Reduce los alargamientos (`jajajaja` → `jaja`).

**No** se quitan stopwords ni se hace stemming, porque el género gramatical (*hijo*/*hija*) y los pronombres (*tú eres*) son informativos para TR.

In [ ]:
ej = d.train['text'].iloc[[2, 21, 80, 668]]
display(pd.DataFrame({'original': ej.values, 'normalizado': ej.map(tb.normalizar).values}))

## 3. Representación y variables lingüísticas
- **Bloques de texto:**
  - TF-IDF de palabras (1-2 gramas).
  - TF-IDF de caracteres `char_wb` (2-5).
  - Bloque de variables lingüísticas: estandarizado y con peso 0,3.
  - Para los modelos de árboles, TruncatedSVD(300) de los TF-IDF más las variables sin escalar.
- **Variables lingüísticas:** son patrones regex escritos por el equipo a partir del análisis del train.
  - **H1-H11** (notebook original): insultos femeninos, *puta* como intensificador, *hijo* frente a *hija de puta*, *cállate* con o sin insulto, grupos migrantes, mención al inicio, comillas...
  - **H12-H17**, versión genérica (E10b): posesivo + insulto (*mi perra*), negación (*no es una zorra*), *árabe* en un estereotipo, grupo migrante + deshumanización o expulsión, demostrativo + grupo (*estos indocumentados*), demostrativo femenino (*esta puta*).
  - **H19-H23** (E13): se diseñaron leyendo **solo** el 80 % de desarrollo; el 20 % de control se dejó sin mirar (sección 6).
    - *perra* como mascota.
    - Estereotipo sexista sin insulto (*feminazi*, *a fregar*).
    - "racismo es que...".
    - *puta* aplicada a algo no humano (*puta serie*, *ni puta idea*).

La tabla compara P(etiqueta | patrón) con la prevalencia. Las variables H19-H23 se calculan **solo en desarrollo**, que es como se decidieron.

In [ ]:
hold = np.load('recursos/control_E13.npy'); dev = np.setdiff1d(np.arange(d.n), hold)
def tabla_patrones(fn_cols, idx):
    filas = {}
    for nombre, (fn, cols) in fn_cols.items():
        M = np.array([fn(r, n) for r, n in zip(d.train['text'].iloc[idx], d.TXT_TR.iloc[idx])])
        for j, c in enumerate(cols):
            m = M[:, j] > 0
            filas[(nombre, c)] = {'n': int(m.sum()), **{f'P({l})': round(d.Y[idx][m, k].mean(), 2) if m.any() else np.nan
                                                       for k, l in enumerate(tb.LABELS)}}
    return pd.DataFrame(filas).T
_, fg, cg = ev.variables(generica=True); _, f2, c2 = ev2.variables()
print('Prevalencia en el train:', dict(zip(tb.LABELS, d.Y.mean(0).round(2))))
display(tabla_patrones({'H12-H17 genérica (todo el train)': (fg, cg)}, np.arange(d.n)))
print('Prevalencia en desarrollo:', dict(zip(tb.LABELS, d.Y[dev].mean(0).round(2))))
display(tabla_patrones({'H19-H23 (solo desarrollo)': (f2, c2)}, dev))

## 4. Diseño de validación
- **CV estratificada** de 5 particiones por la combinación HS-TR-AG, **repetida con 3 semillas** (42, 43, 44). Se reporta la media y la desviación entre semillas.
- **Métrica:** F1 de la clase positiva de cada etiqueta, promediado sobre HS, TR y AG (*F1 medio*).
- **Stacking:** un metamodelo (regresión logística, `C=1`, `class_weight='balanced'`, una por etiqueta) aprende de las probabilidades OOF de los modelos base. Se evalúa con otra CV (semilla + 1), así que tampoco ve las filas que predice.
- ***Seed bagging*:** cada modelo base se entrena con 5 semillas de partición (42-46) y se promedian sus OOF. En el test se promedian los 25 modelos de partición de cada base.
- **Criterio de adopción:** un cambio se adopta si sube el F1 medio ≥ 0,003 y mejora en al menos 2 de las 3 semillas.
- **¿Por qué no se decide con el leaderboard público?** El público usa unas 577 filas, así que el ruido de muestreo del F1 ronda ±2,5 puntos. Eso es más que todas las diferencias entre las alternativas.

## 5. Comparación de alternativas
Cada fila es una alternativa probada en los experimentos E0-E13. Todas se evalúan con la misma CV repetida.

In [ ]:
def stack(dd, bases, bag=True):
    if bag:
        return {s: tb.stacking_bagged(dd, bases, S5, s, con_test=False)[0] for s in SEEDS_CV}
    return {s: tb.stacking(dd, bases, s, con_test=False)[0] for s in SEEDS_CV}
dg = tb.Datos(extra_lex=[ev.variables(generica=True)])
d13 = tb.Datos(extra_lex=[ev.variables(generica=True), ev2.variables()])
R = {}
for b in B7:
    R[f'Base individual · {b}'] = tb.evaluar(d, {s: tb.oof(d, b, s) for s in SEEDS_CV})
    tb.log('base', b)
R['Ensamble LR + Powerset (original v1)'] = tb.evaluar(d, {s: (tb.oof(d, 'LR', s) + tb.oof(d, 'Powerset', s)) / 2 for s in SEEDS_CV})
R['E0 · stacking 5 modelos (original)'] = tb.evaluar(d, stack(d, B5, bag=False))
R['E6 · + seed bagging'] = tb.evaluar(d, stack(d, B5))
R['E8b · + LightGBM'] = tb.evaluar(d, stack(d, B5 + ['LGBM']))
ST = {'E11c': stack(d, B7), 'E11b': stack(dg, B7), 'E13a': stack(d13, B7)}
R['E11c · + LGBM-PS'] = tb.evaluar(d, ST['E11c'])
R['E11b · + H12-H17 genérica'] = tb.evaluar(d, ST['E11b'])
R['E13a · + H19-H23'] = tb.evaluar(d, ST['E13a'])
tabla = tb.tabla(R); tabla

**Lectura de la tabla.**
- Los mejores modelos individuales son Powerset y LR. El stacking les suma unos 0,8 puntos.
- El *seed bagging* y las dos bases de árboles (LightGBM y LGBM-PS) suman otro punto, sobre todo porque reducen la varianza.
- La mayor ganancia viene de las **variables lingüísticas de contexto**. En la CV completa, E11b mejora a E11c y E13a mejora a E11b.

Las alternativas que se probaron y **no** se adoptaron están documentadas en sus notebooks E1-E12 y en `INFORME_EXPERIMENTOS.md`. Ninguna superó el criterio de adopción.
- E1: umbrales por etiqueta con CV anidada.
- E2: reglas posteriores a la predicción.
- E3: más bases lineales.
- E4: Word2Vec entrenado desde cero.
- E5: el léxico HurtLex.
- E7: ajuste de `C` y `min_df`.
- E10a: redes neuronales entrenadas desde cero.
- E10d: modelos de lenguaje Kneser-Ney por clase.
- E12: promedios de stacks y bases extra.

## 6. Validación en un conjunto de control (sesgo de selección)
Las variables escritas mirando los errores del train inflan la CV, porque se evalúan con las mismas filas que inspiraron los patrones. Para medirlo, en E13:
- Se reservó un **20 % del train como control** (347 filas, `StratifiedKFold(5, random_state=2026)`, partición 0, guardado en `recursos/control_E13.npy`).
- Las variables H19-H23 se diseñaron leyendo solo los errores del 80 % restante.

Si la mejora fuera solo sesgo de selección, en el control no aparecería.

In [ ]:
def f1_en(P_por_seed, idx):
    v = [tb.metricas(d.Y[idx], tb.binarizar(P)[idx])['F1 medio'] for P in P_por_seed.values()]
    return f'{np.mean(v):.4f} ({" / ".join(f"{x:.3f}" for x in v)})'
pd.DataFrame({k: {'CV completa': f1_en(P, np.arange(d.n)), 'Desarrollo (80 %)': f1_en(P, dev), 'Control (20 %, no visto)': f1_en(P, hold)}
              for k, P in ST.items()}).T

- E13a supera a E11b en el control, en las 3 semillas. La mejora en desarrollo no es mayor que en el control, así que en H19-H23 no se ve sesgo de selección.
- El control es pequeño (ruido de unos ±0,5 puntos), así que la estimación es indicativa.
- H12-H17 se escribieron mirando todo el train, control incluido, así que su número en el control no es una medida limpia.

## 7. Análisis de errores (OOF de E13a)

In [ ]:
P13 = np.mean(list(ST['E13a'].values()), 0); Yp = tb.binarizar(P13); Yp[:, 1] *= Yp[:, 0]
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
filas = {}
for j, l in enumerate(tb.LABELS):
    tn, fp, fn, tp = confusion_matrix(d.Y[:, j], Yp[:, j]).ravel()
    filas[l] = dict(precision=round(precision_score(d.Y[:, j], Yp[:, j]), 3), recall=round(recall_score(d.Y[:, j], Yp[:, j]), 3),
                    F1=round(f1_score(d.Y[:, j], Yp[:, j]), 3), VP=tp, FP=fp, FN=fn, VN=tn)
display(pd.DataFrame(filas).T)
real = pd.Series(d.COMBO); pred = pd.Series([''.join(map(str, r)) for r in Yp])
err = (real != pred)
print('Errores de combinación más frecuentes (real → predicho):')
display((real[err] + ' → ' + pred[err]).value_counts().head(8).to_frame('n').T)

In [ ]:
def ejemplos(r, p, k=5):
    m = (real == r) & (pred == p)
    return d.train.loc[m, ['text']].assign(P_HS_TR_AG=[' '.join(f'{x:.2f}' for x in P13[i]) for i in np.where(m)[0]]).head(k)
print('Falsos positivos 000 → 111 (insulto femenino sin odio):'); display(ejemplos('000', '111'))
print('Falsos negativos 101 → 000 (odio xenófobo implícito, sin insultos):'); display(ejemplos('101', '000'))
print('Falsos negativos 111 → 000 (insultos en construcciones poco frecuentes y acoso sexual):'); display(ejemplos('111', '000'))

**Patrones de error y limitaciones.**
- **Ruido de anotación.** Construcciones casi idénticas (*cállate perra*) aparecen con etiquetas `000` y `110`. Eso pone un techo a cualquier modelo de bolsa de palabras.
- **Insulto femenino sin odio** (falsos `111`), que es el error más frecuente:
  - El blanco es un animal (*la perra*) o una cosa (*puta serie*).
  - Es un trato entre amigas, o una cita o crítica del insulto (*"¿que las mujeres son perras? no..."*).
  - H12, H13, H19 y H23 atacan este caso, pero sin semántica no se resuelve del todo.
- **Odio xenófobo implícito** (falsos `000`): estereotipos sin insultos (*"me da miedo cuando me sigue un árabe"*, *"como siempre las subvenciones serán para inmigrantes"*). Comparten vocabulario con titulares de prensa anotados `000`.
- **Acoso sexual sin palabras clave:** contenido sexual explícito dirigido que el modelo no marca. Se probó como variable (H18) y se descartó, porque en desarrollo la mayoría de esos tweets **no** es odio (P(HS) ≈ 0,2-0,4).
- **Cambio de distribución hacia el test.** En el leaderboard público, las variantes con más variables de contexto quedaron por debajo de E11c (sección 8). Puede ser ruido, porque difieren en solo 26-56 filas, o que esas variables generalicen peor al test. En HatEval 2019 los organizadores reportaron una caída fuerte de los sistemas entre el desarrollo y el test.

## 8. Submissions finales
Los modelos de test se entrenan con 4/5 del train en cada partición y semilla (`pred_test_folds`), y se promedian. El metamodelo se entrena con todos los OOF del train. Después se aplica:
- La restricción **TR ⊂ HS** (si HS = 0, TR = 0).
- `000` para las filas `#¿NOMBRE?`, porque no tienen texto recuperable.

Se verifica que el resultado coincide fila por fila con los archivos enviados a Kaggle.

In [ ]:
FINALES = {'toxic_submission_E11c_base_lgbmps.csv': d, 'toxic_submission_E13a_generica_h19_23.csv': d13}
for archivo, dd in FINALES.items():
    print('=' * 25, archivo)
    enviado = pd.read_csv(os.path.join('submissions', archivo))
    _, Pt = tb.stacking_bagged(dd, B7, S5, 42)
    sub = tb.guardar_submission(dd, tb.binarizar(Pt), archivo)
    assert (sub.values == enviado.values).all(), 'no coincide con lo enviado'
    print('Coincide exactamente con el archivo enviado a Kaggle (', len(sub), 'filas )')

### Resultados en Kaggle y estrategia de selección

| Submission | CV (F1 medio) | Público |
|---|---|---|
| Stacking original (`../toxic_submission.csv`) | 0,8126 | 0,79160 |
| E8b · bagging + LightGBM | 0,8179 | 0,80168 |
| **E11c** · + LGBM-PS | 0,8204 | **0,80206** |
| E11b · + H12-H17 genérica | 0,8244 | 0,79895 |
| E11a · + H12-H17 completa | 0,8300 | 0,79644 |
| **E13a** · + H12-H17 genérica + H19-H23 | **0,8284** | 0,79597 |

**Estrategia.** Las decisiones se toman con la CV repetida y no con el público, porque el ruido del público (±2,5 puntos) es mayor que las diferencias. Kaggle califica el privado con la mejor de las submissions marcadas, así que se marcan las dos que cubren hipótesis distintas:
- **E11c:** sin variables de contexto nuevas. Es el mejor público, con una CV no sesgada por el análisis de errores.
- **E13a:** con las variables de contexto. Es la mejor CV entre las variantes con variables, y la única con mejora medida en un control no visto.

## 9. Referencias y licencias
- Basile, V. et al. (2019). *SemEval-2019 Task 5: Multilingual Detection of Hate Speech Against Immigrants and Women in Twitter (HatEval)*. Corpus de la competencia. Su versión etiquetada **no** se usó como dato adicional, porque el test sale de ahí.
- Wang, S. y Manning, C. (2012). *Baselines and Bigrams: Simple, Good Sentiment and Topic Classification* (NB-LR).
- Ke, G. et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision Tree*. NeurIPS. Licencia MIT.
- Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python*. JMLR. Licencia BSD-3.
- Bassignana, E., Basile, V. y Patti, V. (2018). *HurtLex*. CC BY-NC-SA 4.0. Solo se usó en E5 y no se adoptó.
- No se usó ningún modelo, embedding ni transformer preentrenado.